<a href="https://colab.research.google.com/github/samer-glitch/TADP-Enhancing-Trustworthiness-in-Federated-Learning-Through-Provenance-Driven-Governance/blob/main/2_Fashion_MNIST.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


In [4]:

from google.colab import drive

drive.mount('/content/drive')


Mounted at /content/drive


In [5]:

from pathlib import Path

print("Drive mounted:", Path('/content/drive/MyDrive').is_dir())


Drive mounted: True


In [6]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""SELF-CONTAINED COLAB SCRIPT: TADP-HPS-28 / phased multi-seed FL benchmark

Paste this entire file into a Google Colab code cell and run it. No other
TADP Python modules or ZIP extraction required. Requires standard scientific
Python libraries (NumPy, pandas, PyTorch, scikit-learn, SciPy, matplotlib).

Methodology: HPS D1-D6 (28 factors; 8 TRAIN-derived D2, 20 documentary
rubric factors). The 20 documentary levels are FROZEN SYNTHETIC PROFILES,
not actual human-verified records. The human evidence-checking step is not
automated by this script.

This is a RESEARCH-GRADE benchmark, not a production clinical/governance
service: reviewers are simulated, no cryptographic attestation, no external
ledger anchoring, no secure aggregation, and no networked FL deployment.
No seed or threshold may be tuned on held-out TEST. Adult lacks patient IDs:
its split uses exact-duplicate predictor grouping, not patient grouping.
The TRAIN-only governance distribution reference is never model preprocessing.
The pilot is intended to check execution, NOT to support publication claims.
Publication claims require complete matched-seed runs and independent review of results.
"""
from __future__ import annotations
from IPython.display import display
import matplotlib.pyplot as plt
from urllib.request import urlretrieve
import traceback
import platform
import sys



# ==============================================================================
# EMBEDDED MODULE: hps_policy.py
# ==============================================================================
"""TADP: 28-factor Hybrid Provenance Score (HPS) admission policy.

Documentary levels are HUMAN-ASSESSED in deployment; SIMULATED evidence
profiles in these experiments must not be presented as human verification.
"""
from dataclasses import dataclass, field, asdict
from math import isfinite
from numbers import Real
from typing import Mapping, Any

POLICY_VERSION = "TADP-HPS-28/1.0"
FACTORS = {
    "D1": ("source_reputation", "data_controller", "data_objective", "data_collection_lineage"),
    "D2": ("completeness", "duplication_rate", "value_validity_error_rate", "type_consistency",
           "label_integrity", "feature_distribution_consistency", "feature_category_coverage",
           "structural_constraint_integrity"),
    "D3": ("data_dictionary", "version_logs", "collection_protocol", "definition_updates"),
    "D4": ("data_freshness", "scheduled_refresh", "retention_clarity"),
    "D5": ("regulation_coverage", "consent_ethics", "geo_restrictions", "sensitivity_classification", "audits"),
    "D6": ("license_terms", "ethical_reviews", "redistribution", "user_agreements"),
}
WEIGHTS = {"D1": .25, "D2": .15, "D3": .10, "D4": .10, "D5": .30, "D6": .10}
CRITICAL = ("D1", "D2", "D5", "D6")
ALL_FACTORS = frozenset(f for dim in FACTORS.values() for f in dim)
DOC_FACTORS = frozenset(f for d, fs in FACTORS.items() if d != "D2" for f in fs)
D2_FACTORS = FACTORS["D2"]
assert len(ALL_FACTORS) == 28 and len(DOC_FACTORS) == 20 and len(D2_FACTORS) == 8
assert abs(sum(WEIGHTS.values()) - 1) < 1e-12

@dataclass(frozen=True)
class Policy:
    min_dimension: float = 2.5
    reject_below: float = 3.0
    direct_accept_from: float = 3.5
    # Explicit individual critical-factor safeguards, if legally/operationally justified.
    critical_factor_floors: Mapping[str, int] = field(default_factory=dict)
    policy_version: str = POLICY_VERSION

    @property
    def review_midpoint(self):
        return (self.reject_below + self.direct_accept_from) / 2.0

    def validate(self):
        bounds = (self.min_dimension, self.reject_below, self.direct_accept_from)
        if not all(isfinite(v) and 0 <= v <= 5 for v in bounds):
            raise ValueError("Invalid policy thresholds")
        if not (self.min_dimension <= self.reject_below < self.direct_accept_from):
            raise ValueError("Require minimum dimension <= reject threshold < accept threshold")
        for f, threshold in self.critical_factor_floors.items():
            if f not in ALL_FACTORS or type(threshold) is not int or not 0 <= threshold <= 5:
                raise ValueError(f"Invalid critical-factor floor: {f}")

@dataclass(frozen=True)
class Decision:
    accepted: bool
    path: str
    hps: float | None
    hps_review: float | None
    dimensions: dict[str, float]
    reasons: tuple[str, ...]
    factor_scores: dict[str, int]
    policy_version: str

    def record(self) -> dict:
        return asdict(self)


def evaluate(scores: Mapping[str, int], policy: Policy = Policy(),
             disqualifiers: tuple[str, ...] = (),
             ablate_distribution: bool = False) -> Decision:
    policy.validate()
    missing = sorted(ALL_FACTORS - scores.keys())
    unknown = sorted(scores.keys() - ALL_FACTORS)
    if missing or unknown:
        return Decision(False, "BLOCKED_INCOMPLETE", None, None, {},
                        (f"Missing={missing}, unexpected={unknown}",), dict(scores), policy.policy_version)
    for factor, value in scores.items():
        if type(value) is not int or not 0 <= value <= 5:
            return Decision(False, "BLOCKED_INVALID", None, None, {},
                            (f"Invalid rubric score: {factor}={value!r}",), dict(scores), policy.policy_version)
    factor_groups = dict(FACTORS)
    if ablate_distribution:
        factor_groups["D2"] = tuple(f for f in FACTORS["D2"]
                                     if f not in ("feature_distribution_consistency","feature_category_coverage"))
    dimensions = {d: sum(scores[f] for f in fs)/len(fs) for d, fs in factor_groups.items()}
    hps = sum(WEIGHTS[d]*dimensions[d] for d in FACTORS)
    den = sum(WEIGHTS[d] for d in CRITICAL)
    hps_review = sum(WEIGHTS[d]*dimensions[d] for d in CRITICAL)/den
    def out(accepted, path, *reason):
        version = policy.policy_version + ("/NoDistributionAblation" if ablate_distribution else "")
        return Decision(accepted, path, round(hps, 10), round(hps_review, 10),
                        dimensions, tuple(reason), dict(scores), version)
    if disqualifiers:
        return out(False, "BLOCKED_DISQUALIFIER", *disqualifiers)
    active = set(f for fs in factor_groups.values() for f in fs)
    zeros = sorted(f for f, v in scores.items() if f in active and v == 0)
    if zeros:
        return out(False, "BLOCKED_ZERO_FACTOR", f"Required factor(s) scored zero: {zeros}")
    low = [f"{d}={v:.3f} < {policy.min_dimension}" for d, v in dimensions.items()
           if v < policy.min_dimension]
    if low:
        return out(False, "BLOCKED_DIMENSION_FLOOR", *low)
    for f, floor in policy.critical_factor_floors.items():
        if scores[f] < floor:
            return out(False, "BLOCKED_CRITICAL_FACTOR", f"{f}={scores[f]} < {floor}")
    if hps < policy.reject_below:
        return out(False, "AUTO_REJECT", f"HPS={hps:.4f} < {policy.reject_below}")
    if hps >= policy.direct_accept_from:
        return out(True, "AUTO_ACCEPT", f"HPS={hps:.4f} >= {policy.direct_accept_from}")
    if hps_review >= policy.review_midpoint:
        return out(True, "REVIEW_ACCEPT", f"HPS_review={hps_review:.4f} >= {policy.review_midpoint}")
    return out(False, "REVIEW_REJECT", f"HPS_review={hps_review:.4f} < {policy.review_midpoint}")


# Proportions with >lower boundary map down one category; all values must lie in [0,1].
_VIOLATION_CUTS = {
    "completeness": (.50, .20, .10, .05, .01),
    "duplication_rate": (.20, .10, .05, .02, .01),
    "value_validity_error_rate": (.15, .10, .05, .02, .01),
    "type_consistency": (.20, .10, .05, .02, .01),
    "label_integrity": (.10, .05, .02, .01, .001),
    "feature_distribution_consistency": (.20, .10, .05, .025, .01),
    "structural_constraint_integrity": (.10, .05, .02, .01, .001),
}

def d2_from_metrics(metrics: Mapping[str, float]) -> dict[str, int]:
    if set(metrics) != set(D2_FACTORS):
        raise ValueError(f"Expected all eight D2 factors, missing={set(D2_FACTORS)-set(metrics)}")
    result = {}
    for name, value in metrics.items():
        if isinstance(value, bool) or not isinstance(value, Real) or not isfinite(value) or not 0 <= value <= 1:
            raise ValueError(f"Invalid TRAIN-only metric {name}={value!r}")
        if name == "feature_category_coverage":
            if value < .50: score = 0
            elif value < .65: score = 1
            elif value < .75: score = 2
            elif value < .825: score = 3
            elif value < .90: score = 4
            else: score = 5
        else:
            score = next((i for i, cut in enumerate(_VIOLATION_CUTS[name]) if value > cut), 5)
        result[name] = score
    return result


def verified_document_scores(records: Mapping[str, Mapping[str, Any]]) -> dict[str, int]:
    """Reviewers select a rubric level supported by inspected evidence.

    'synthetic' is accepted solely to run labeled experimental simulations.
    This function does NOT independently authenticate documentary evidence.
    """
    if set(records) != DOC_FACTORS:
        raise ValueError("Missing/unexpected documentary factors")
    scores = {}
    for f, record in records.items():
        if record.get("mode") not in ("human", "synthetic") or record.get("verified") is not True:
            raise ValueError(f"Unverified evidence: {f}")
        if not record.get("reviewer_id"):
            raise ValueError(f"Missing reviewer for {f}")
        level = record.get("rubric_level")
        if type(level) is not int or not 0 <= level <= 5:
            raise ValueError(f"Invalid reviewer level: {f}")
        if level > 0 and not record.get("evidence_ref"):
            raise ValueError(f"Missing evidence reference for {f}")
        scores[f] = level
    return scores


def evaluate_client(document_records, train_quality_metrics, policy=Policy(), disqualifiers=()):
    return evaluate({**verified_document_scores(document_records),
                     **d2_from_metrics(train_quality_metrics)}, policy, disqualifiers)


# ==============================================================================
# EMBEDDED MODULE: hps_evidence.py
# ==============================================================================
"""Factor-specific evidence templates drawn from Appendix A; synthetic experimental use only."""
import hashlib
import json
from pathlib import Path
from random import Random

RUBRICS = {
 "source_reputation": ("No information", "Poor", "Limited evidence", "Average, partially trusted", "Well-documented, reliable", "Highly reputable, verified"),
 "data_controller": ("No documented controller", "Unclear", "Partially clear", "Moderately clear", "Mostly clear", "Fully documented"),
 "data_objective": ("None", "Vague", "Partial", "General but unclear", "Mostly explicit", "Fully explicit and justified"),
 "data_collection_lineage": ("Origin unknown", "Informal/unverifiable origin", "Partially documented path", "Documented with limited traceability", "Well-documented traceable path", "Fully source-linked/versioned/auditable"),
 "data_dictionary": ("None", "Minimal outline", "Partial coverage", "Moderate coverage", "Near-complete", "Fully detailed"),
 "version_logs": ("None", "Minimal logs", "Occasional logs", "Regular logs", "Near-complete", "Full version history"),
 "collection_protocol": ("None", "Vague", "Partial", "General methods", "Well-defined", "Fully transparent"),
 "definition_updates": ("None", "Rare", "Occasional", "Regular but basic", "Frequent", "Real-time documented"),
 "data_freshness": (">5 years", "2-5 years", "1-2 years", "6-12 months", "1-6 months", "Current"),
 "scheduled_refresh": ("Never", "Irregular", "Annual", "Quarterly", "Monthly", "Daily/real time"),
 "retention_clarity": ("None", "Minimal", "Basic guidelines", "Moderate clarity", "High clarity", "Fully documented"),
 "regulation_coverage": ("None", "Minimal", "Partial", "Moderate", "Comprehensive but dated", "Fully documented/current"),
 "consent_ethics": ("None", "Minimal record", "Partial", "Moderate logs", "Substantial", "Fully documented"),
 "geo_restrictions": ("None", "Basic mention", "Partial", "Moderate", "Near-complete", "Fully documented"),
 "sensitivity_classification": ("None", "Basic flagging", "Partial", "Moderate", "Near-complete", "Fully classified"),
 "audits": ("None", "Internal only", "Basic certification", "Occasional audit", "Recent audit", "Regular external audits"),
 "license_terms": ("None", "Vague", "Basic", "Clear", "Detailed", "Industry-compliant"),
 "ethical_reviews": ("None", "Informal", "Partial", "Moderate", "Well-documented", "Certified"),
 "redistribution": ("None", "Unclear", "Partial", "Clear", "Detailed", "Fully documented"),
 "user_agreements": ("Non-compliant", "Minimal", "Partial", "Mostly compliant", "Fully compliant", "Audited"),
}
assert set(RUBRICS) == DOC_FACTORS and all(len(x) == 6 for x in RUBRICS.values())

# Profiles are correlated across factors to avoid implausibly incoherent random vectors.
# The profile assignment itself is a synthetic assumption; NOT independently verified.
BANDS = {"high": (4, 5), "medium": (3, 4), "low": (1, 2), "missing": (0, 1)}

def make_manifest(ids: list[str], seed: int, profile_map: dict | None = None):
    rng = Random(seed)
    if len(set(ids)) != len(ids): raise ValueError("Duplicate IDs")
    if profile_map is not None and set(profile_map) != set(ids): raise ValueError("Incomplete profile map")
    clients = {}
    for cid in sorted(ids):
        tier = profile_map[cid] if profile_map is not None else rng.choices(
            list(BANDS), weights=[.52, .33, .10, .05])[0]
        lo, hi = BANDS[tier]
        doc = {}
        for factor in sorted(DOC_FACTORS):
            score = rng.randint(lo, hi)
            doc[factor] = {"rubric_level": score, "descriptor": RUBRICS[factor][score],
                            "mode": "synthetic", "verified": True,
                            "evidence_ref": (f"SIMULATED:{cid}:{factor}:L{score}" if score>0 else ""),
                            "reviewer_id": "SYNTHETIC_PROFILE_NOT_HUMAN_REVIEW"}
        clients[cid] = {"tier": tier, "documentary_factors": doc}
    body = {"policy_version": POLICY_VERSION, "synthetic": True, "seed": seed, "clients": clients}
    body["sha256"] = _digest(body)
    return body

def _digest(obj):
    return hashlib.sha256(json.dumps({k:v for k,v in obj.items() if k != "sha256"},
                                     sort_keys=True, separators=(",", ":")).encode()).hexdigest()

def save_manifest(body, path):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text(json.dumps(body, indent=2, sort_keys=True), encoding="utf-8")

def load_manifest(path):
    obj = json.loads(Path(path).read_text(encoding="utf-8"))
    if obj["sha256"] != _digest(obj): raise ValueError("Evidence manifest was modified")
    return obj


# ==============================================================================
# EMBEDDED MODULE: hps_data_quality.py
# ==============================================================================
"""Eight D2 checks computed from RAW client TRAIN data only.

Two distribution factors compare with a FROZEN TRAIN-only calibration set.
This comparison measures representativeness, not corruption; preserve its
impact in an ablation. Label flips that remain valid are not observable without
independently audited ground truth and are intentionally NOT inferred here.
"""
import hashlib
import numpy as np
import pandas as pd
from scipy.stats import ks_2samp

ADULT_NUMERIC = ("age", "fnlwgt", "education-num", "capital-gain", "capital-loss", "hours-per-week")
NUMERIC_RANGES = {"age": (1, 120), "fnlwgt": (1, 1e10), "education-num": (1, 16),
                  "capital-gain": (0, 1e9), "capital-loss": (0, 1e9), "hours-per-week": (0, 168)}

def _ks_max(a, b, seed=1, cap=10000):
    rng = np.random.default_rng(seed)
    a, b = np.asarray(a).reshape(-1), np.asarray(b).reshape(-1)
    a, b = a[np.isfinite(a)], b[np.isfinite(b)]
    if not len(a) or not len(b): return 1.0
    if len(a) > cap: a = rng.choice(a, cap, replace=False)
    if len(b) > cap: b = rng.choice(b, cap, replace=False)
    return float(ks_2samp(a, b).statistic)

def adult_train_metrics(client_x: pd.DataFrame, labels, reference_x: pd.DataFrame):
    if client_x.empty: raise ValueError("No client training records")
    if not set(ADULT_NUMERIC) <= set(client_x.columns): raise ValueError("Adult schema mismatch")
    if set(client_x.columns) != set(reference_x.columns): raise ValueError("Reference schema mismatch")
    x = client_x.replace(r"^\s*\?\s*$", np.nan, regex=True).replace(r"^\s*$", np.nan, regex=True)
    ref = reference_x.replace(r"^\s*\?\s*$", np.nan, regex=True).replace(r"^\s*$", np.nan, regex=True)
    numeric, categorical = list(ADULT_NUMERIC), [c for c in x.columns if c not in ADULT_NUMERIC]
    n, c = x.shape
    completeness = float(x.isna().to_numpy().mean())
    duplication = float(x.duplicated().mean())
    numeric_arr = x[numeric].apply(pd.to_numeric, errors="coerce")
    num_observed = x[numeric].notna().to_numpy()
    numeric_type_invalid = np.logical_and(num_observed, numeric_arr.isna().to_numpy()).sum()
    # Expected categorical fields are strings (allow object/string and reject unexpected list-like data).
    cat_type_bad = sum(sum(not isinstance(v, (str, np.str_)) for v in x[col].dropna()) for col in categorical)
    observed_count = int(x.notna().to_numpy().sum())
    type_inconsistency = (numeric_type_invalid + cat_type_bad) / max(observed_count, 1)
    num_invalid = np.zeros((n, len(numeric)), dtype=bool)
    bad_row = np.zeros(n, dtype=bool)
    for idx, field in enumerate(numeric):
        vals = numeric_arr[field].to_numpy(dtype=float)
        lo, hi = NUMERIC_RANGES[field]
        bad = np.isfinite(vals) & ((vals < lo) | (vals > hi))
        num_invalid[:, idx] = bad
        bad_row |= bad
    # Numeric type errors also count as invalid values, independently of completeness.
    validity = float((num_invalid.sum() + numeric_type_invalid) / max(observed_count, 1))
    structural = float(bad_row.mean())
    ys = np.asarray(labels)
    if len(ys) != n: raise ValueError("Label count mismatch")
    label_bad = float((~np.isin(ys, [0, 1])).mean())
    ks_vals = []
    for k, field in enumerate(numeric):
        r = pd.to_numeric(ref[field], errors="coerce").to_numpy(dtype=float)
        p = numeric_arr[field].to_numpy(dtype=float)
        ks_vals.append(_ks_max(p, r, seed=1337+k))
    consistency = max(ks_vals)
    coverage = []
    for field in categorical:
        ref_levels = set(ref[field].dropna().astype(str).str.strip().str.lower())
        present = set(x[field].dropna().astype(str).str.strip().str.lower())
        if ref_levels: coverage.append(len(ref_levels & present)/len(ref_levels))
    return dict(completeness=completeness, duplication_rate=duplication,
                value_validity_error_rate=min(1.,validity), type_consistency=min(1.,float(type_inconsistency)),
                label_integrity=label_bad, feature_distribution_consistency=float(consistency),
                feature_category_coverage=float(np.mean(coverage) if coverage else 1),
                structural_constraint_integrity=structural)

def _sample_hashes(images):
    return [hashlib.blake2b(np.ascontiguousarray(row).tobytes(), digest_size=8).digest()
            for row in images]

def fashion_train_metrics(images: np.ndarray, labels, reference_images: np.ndarray):
    x = np.asarray(images)
    if x.ndim != 3 or x.shape[1:] != (28, 28): raise ValueError("Fashion input expected (N,28,28)")
    n = x.shape[0]
    if not n: raise ValueError("No images")
    y = np.asarray(labels)
    if len(y) != n: raise ValueError("Label count mismatch")
    observed = np.isfinite(x)
    missing = 1.0 - float(observed.mean())
    duplication = 1 - len(set(_sample_hashes(x))) / n
    # Raw images stored on 0..255 scale. Pixel errors share some constraints
    # with structural checks; this overlap is documented, not ignored.
    out_of_range = observed & ((x < 0) | (x > 255))
    invalid = float(out_of_range.sum() / max(observed.sum(),1))
    type_errors = observed & (x != np.floor(x))
    dtype_bad = float(type_errors.sum() / max(observed.sum(),1))
    structural = float(out_of_range.reshape(n,-1).any(axis=1).mean())
    label_bad = float((~np.isin(y, np.arange(10))).mean())
    distribution = _ks_max(x, np.asarray(reference_images), seed=2026, cap=15000)
    # 16 fixed raw-pixel categories [0,16), ..., [240,256).
    valid_pixels = x[observed & (x >= 0) & (x <= 255)]
    ref = np.asarray(reference_images)
    reference_pixels = ref[np.isfinite(ref) & (ref >= 0) & (ref <= 255)]
    observed_bins = set(np.clip((valid_pixels // 16).astype(int),0,15).tolist())
    ref_bins = set(np.clip((reference_pixels // 16).astype(int),0,15).tolist())
    cover = len(observed_bins & ref_bins) / max(len(ref_bins),1)
    return dict(completeness=missing, duplication_rate=float(duplication),
                value_validity_error_rate=invalid, type_consistency=dtype_bad,
                label_integrity=label_bad, feature_distribution_consistency=distribution,
                feature_category_coverage=float(cover), structural_constraint_integrity=structural)


def inject_degradation(x, y, dataset: str, rng: np.random.Generator,
                       missing_fraction=.12, duplicate_fraction=.15,
                       invalid_fraction=.06, label_flip_fraction=.20):
    """Only mutates a COPY of selected CLIENT TRAIN data, never calibration/TEST."""
    y = np.asarray(y).copy()
    n = len(y)
    if dataset == "adult":
        z = x.copy(deep=True)
        if n > 2:
            dup_ids = rng.choice(n, max(1,int(n*duplicate_fraction)), replace=False)
            origin = rng.choice(n, len(dup_ids), replace=True)
            z.iloc[dup_ids] = z.iloc[origin].to_numpy()
            y[dup_ids] = y[origin]
        rows = rng.integers(0,n, size=max(1,int(n*len(z.columns)*missing_fraction)))
        cols = rng.integers(0,len(z.columns),size=len(rows))
        for r,c in zip(rows,cols): z.iat[int(r),int(c)] = np.nan
        numcol = "hours-per-week"
        invalid_ids = rng.choice(n,max(1,int(n*invalid_fraction)), replace=False)
        z.iloc[invalid_ids, z.columns.get_loc(numcol)] = -999
        flips = rng.choice(n,max(1,int(n*label_flip_fraction)),replace=False)
        y[flips] = 1-y[flips]
        return z,y
    if dataset == "fashion":
        z = np.asarray(x).copy().astype(np.float32)
        if n > 2:
            dup_ids = rng.choice(n,max(1,int(n*duplicate_fraction)),replace=False)
            orig = rng.choice(n,len(dup_ids),replace=True)
            z[dup_ids] = z[orig]
            y[dup_ids] = y[orig]
        missing_count = max(1,int(z.size*missing_fraction))
        idx = rng.choice(z.size,missing_count,replace=False)
        z.flat[idx] = np.nan
        invalid_count = max(1,int(z.size*invalid_fraction))
        idx = rng.choice(z.size,invalid_count,replace=False)
        z.flat[idx] = 999
        flips = rng.choice(n,max(1,int(n*label_flip_fraction)),replace=False)
        y[flips] = (y[flips] + rng.integers(1,10,size=len(flips))) % 10
        return z, y
    raise ValueError(dataset)


# ==============================================================================
# EMBEDDED MODULE: hps_ledger.py
# ==============================================================================
"""Lightweight SHA-256 hash chain; NOT resistant to a server that recomputes history.

To detect server-side rewriting, securely retain/publish signed head hashes
with an INDEPENDENT verifier outside server control. This module does not
implement external anchoring, signatures, or consensus.
"""
import json, hashlib
from pathlib import Path

def _canonical(data):
    return json.dumps(data,sort_keys=True,separators=(",", ":"),allow_nan=False,default=str).encode()

def _hash(data):return hashlib.sha256(_canonical(data)).hexdigest()

def append_event(path, event):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    # Do not trust an existing ledger unless verified before appending.
    ok, _=verify_ledger(path)
    if not ok:raise ValueError("Existing ledger hash chain failed validation")
    entries=[] if not path.exists() else [json.loads(x) for x in path.read_text().splitlines() if x.strip()]
    row={"sequence":len(entries),"prev_hash":entries[-1]["row_hash"] if entries else "GENESIS",
         "event":event}
    row["row_hash"]=_hash(row)
    with path.open("a",encoding="utf-8") as f:f.write(json.dumps(row,sort_keys=True,default=str)+"\n")
    return row["row_hash"]

def verify_ledger(path, trusted_head=None):
    path=Path(path)
    if not path.exists():return (trusted_head is None),"GENESIS"
    previous="GENESIS"
    for i,line in enumerate(path.read_text().splitlines()):
        try:
            r=json.loads(line)
            digest=r.pop("row_hash")
            if r["sequence"]!=i or r["prev_hash"]!=previous or _hash(r)!=digest:return False,previous
            previous=digest
        except (KeyError,ValueError,TypeError,IndexError):return False,previous
    return (trusted_head is None or previous==trusted_head),previous


# ==============================================================================
# EMBEDDED MODULE: tadp_fl.py
# ==============================================================================
"""Reproducible 28-factor HPS federated-learning benchmark on Adult/Fashion-MNIST.

No actual network transfer; model-byte counts are parameter-payload estimates.
No public-data dataset is bundled; Fashion-MNIST may download on first use.
"""
import copy
import json
import math
import random
import time
from dataclasses import dataclass, asdict, replace
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (accuracy_score, f1_score, recall_score, roc_auc_score,
                             balanced_accuracy_score, average_precision_score,
                             precision_score, confusion_matrix)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


ADULT_COLUMNS = ["age", "workclass", "fnlwgt", "education", "education-num", "marital-status",
                 "occupation", "relationship", "race", "sex", "capital-gain", "capital-loss",
                 "hours-per-week", "native-country", "income"]
METHODS = ("FedAvg_all", "FedAvg_matched", "FedProx", "Median", "Krum",
           "PowerOfChoice", "HPG", "PWA", "WUPW", "RandomHPS", "D2_only",
           "NoDist_HPG", "NoDist_PWA")

@dataclass(frozen=True)
class Config:
    dataset: str = "adult"
    n_clients: int = 10
    rounds: int = 15
    per_round: int = 3
    local_steps: int = 10
    batch_size: int = 64
    lr: float = .001
    alpha: float = .5
    warmup_rounds: int = 5
    corruption_fraction: float = .30
    fedprox_mu: float = .01
    krum_f: int = 1
    calibration_fraction: float = .10  # compatibility; no rows withheld for D2 reference
    test_fraction: float = .20
    global_split_seed: int = 2026
    min_client_samples: int = 16
    max_partition_repair_fraction: float = 0.10
    category_hash_bins: int = 32
    use_train_cohort_class_weight: bool = True
    class_weight_cap: float = 5.0
    device: str = "cpu"
    energy_power_kw_assumed: float = .1
    co2_kg_per_kwh_assumed: float = .4

    def validate(self):
        if self.dataset not in ("adult", "fashion"): raise ValueError("Unknown dataset")
        if min(self.n_clients, self.rounds, self.per_round, self.local_steps, self.batch_size) < 1:
            raise ValueError("Nonpositive budget")
        if self.per_round > self.n_clients: raise ValueError("Too many clients per round")
        if not (0 < self.test_fraction < 1 and 0 < self.corruption_fraction <= 1):
            raise ValueError("Invalid data fractions")
        if self.min_client_samples < 1 or self.category_hash_bins < 4 or not 0 <= self.max_partition_repair_fraction <= 1:
            raise ValueError("Invalid fixed partition or feature settings")
        if self.class_weight_cap < 1 or self.warmup_rounds >= self.rounds:
            raise ValueError("Invalid class-weight cap or warm-up")


def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)


def load_dataset(dataset, adult_csv=None, fashion_root="./data", download=True):
    if dataset == "adult":
        if not adult_csv: raise ValueError("Provide --adult-csv path to Adult CSV (raw with missing values)")
        raw = pd.read_csv(adult_csv, skipinitialspace=True, na_values=["?", " ?", "", "nan"])
        if raw.shape[1] == 15 and str(raw.columns[0]).lower() != "age":
            # Headerless UCI adult.data file: preserve first data row.
            raw = pd.read_csv(adult_csv, header=None, skipinitialspace=True,
                              na_values=["?", " ?", "", "nan"])
        if raw.shape[1] != 15: raise ValueError("Adult CSV must have 15 columns")
        if list(raw.columns) != ADULT_COLUMNS:
            if ("income" in [str(c).lower() for c in raw.columns] or len(raw.columns)==15):
                raw.columns = ADULT_COLUMNS
        raw["income"] = raw["income"].astype(str).str.strip().str.lower().str.rstrip(".")
        raw = raw[raw["income"].isin(["<=50k", ">50k"])].copy()
        if raw.empty: raise ValueError("No recognized Adult binary income labels")
        y = (raw["income"] == ">50k").astype(np.int64).to_numpy()
        return raw.drop(columns="income").reset_index(drop=True), y
    try:
        from torchvision.datasets import FashionMNIST
    except Exception as exc:
        raise RuntimeError("torchvision is needed for Fashion-MNIST") from exc
    tr = FashionMNIST(root=fashion_root, train=True, download=download)
    te = FashionMNIST(root=fashion_root, train=False, download=download)
    return (tr.data.numpy(), tr.targets.numpy(), te.data.numpy(), te.targets.numpy())


def split_global(dataset, loaded, cfg: Config, seed):
    """One immutable TRAIN/TEST division across ALL experimental seeds and methods.

    Adult has no patient identifier: claims of patient-grouped splitting are INVALID.
    Instead, identical raw predictor records are grouped to prevent exact
    duplicate predictors occurring in both TRAIN and TEST. Approximate 80/20,
    stratified as closely as possible under those grouping constraints.

    D2-only reference uses GLOBAL TRAIN predictors; it is never fitted as model
    preprocessing and is never used for gradient updates separately.
    """
    if dataset == "adult":
        from sklearn.model_selection import StratifiedGroupKFold
        x, y = loaded
        groups = pd.util.hash_pandas_object(x.fillna("__RAW_MISSING__").astype(str), index=False).to_numpy()
        splits = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=cfg.global_split_seed)
        target_frac = cfg.test_fraction
        global_rate = float(np.mean(y))
        candidates = []
        for train_index, test_index in splits.split(x, y, groups):
            frac = len(test_index) / len(y)
            rate = float(np.mean(y[test_index]))
            candidates.append((abs(frac-target_frac)+abs(rate-global_rate), train_index, test_index))
        _, train_index, test_index = min(candidates, key=lambda z:z[0])
        if set(groups[train_index]) & set(groups[test_index]):
            raise AssertionError("LEAKAGE: identical raw predictor group occurs in TRAIN and TEST")
        xtrain = x.iloc[train_index].reset_index(drop=True)
        xtest = x.iloc[test_index].reset_index(drop=True)
        ytrain, ytest = np.asarray(y)[train_index], np.asarray(y)[test_index]
        assert len(xtrain)+len(xtest)==len(x)
        assert not (set(groups[train_index]) & set(groups[test_index]))
        return xtrain, ytrain, xtrain, xtest, ytest
    xtrain,ytrain,xtest,ytest = loaded
    return xtrain,np.asarray(ytrain),xtrain,xtest,np.asarray(ytest)


def dirichlet_partition(y, n_clients, alpha, seed, min_samples=16, max_repair_fraction=.10,
                        return_diagnostics=False):
    """Seeded label-wise Dirichlet allocation plus EXPLICIT minimum-size repair.

    Rejection sampling with K=50, alpha=0.1 is rarely feasible. Instead sample
    the requested Dirichlet ONCE, then relocate the minimum records required
    from largest donors into undersized clients, preserving class labels.
    Because the repair changes the sampled allocation, report its count/rate;
    never describe the final partition as an unmodified Dirichlet draw.
    """
    rng=np.random.default_rng(seed)
    y=np.asarray(y)
    if alpha<=0 or len(y)<n_clients*min_samples:
        raise ValueError("Impossible partition: alpha<=0 or insufficient TRAIN records")
    bags=[[] for _ in range(n_clients)]
    for label in np.unique(y):
        indices=rng.permutation(np.flatnonzero(y==label))
        proportions=rng.dirichlet(np.full(n_clients, alpha))
        counts=rng.multinomial(len(indices), proportions)
        for i,part in enumerate(np.split(indices, np.cumsum(counts)[:-1])):
            bags[i].extend(part.tolist())
    pre_min=min(map(len,bags))
    moved=0
    for receiver in sorted(range(n_clients),key=lambda i:(len(bags[i]),i)):
        while len(bags[receiver]) < min_samples:
            donors=[j for j in range(n_clients) if len(bags[j])>min_samples]
            if not donors:raise ValueError("Partition quota infeasible; no donor clients")
            donor=max(donors,key=lambda j:(len(bags[j]),-j))
            j=int(rng.integers(len(bags[donor])))
            bags[receiver].append(bags[donor].pop(j));moved+=1
    assigned=np.concatenate([np.asarray(b,dtype=int) for b in bags])
    if len(assigned)!=len(y) or len(set(assigned.tolist()))!=len(y):
        raise AssertionError("Partition has dropped or duplicated TRAIN records")
    repair_fraction=moved/len(y)
    if repair_fraction>max_repair_fraction:
        raise ValueError(f"Partition repair {repair_fraction:.2%} > prespecified cap {max_repair_fraction:.2%}")
    diag={"alpha_requested":float(alpha),"seed":int(seed),"n_clients":n_clients,
          "min_client_size_before_repair":int(pre_min),"min_client_size_after_repair":int(min(map(len,bags))),
          "records_reallocated":int(moved),"repair_fraction":float(repair_fraction),
          "min_samples":int(min_samples),"distribution":"Dirichlet with logged minimum-client-size repair"}
    indices=[np.array(sorted(b),dtype=int) for b in bags]
    return (indices,diag) if return_diagnostics else indices


def make_clients(dataset, x_train, y_train, ids, cfg, seed, degraded=False):
    indices,partition_diag = dirichlet_partition(y_train, cfg.n_clients, cfg.alpha, seed+31,
        min_samples=cfg.min_client_samples,max_repair_fraction=cfg.max_partition_repair_fraction,
        return_diagnostics=True)
    rng=np.random.default_rng(seed+339)
    corrupt_ids=set(rng.choice(ids,size=max(1,math.ceil(cfg.corruption_fraction*len(ids))),
                               replace=False).tolist()) if degraded else set()
    clients={}
    for i,cid in enumerate(ids):
        x = x_train.iloc[indices[i]].reset_index(drop=True) if dataset=="adult" else x_train[indices[i]].copy()
        y = np.asarray(y_train)[indices[i]].copy()
        if cid in corrupt_ids:
            x,y=inject_degradation(x,y,dataset,np.random.default_rng(seed*10007+i))
        clients[cid]={"x_raw":x,"y_raw":y,"n":len(y),"degraded":cid in corrupt_ids}
    return clients,sorted(corrupt_ids),partition_diag


def assess_clients(dataset, clients, xreference, manifest, policy, threat="none"):
    decisions, metric_map, records = {}, {}, {}
    rng = np.random.default_rng(205)
    target_low = min(clients, key=lambda c: manifest["clients"][c]["documentary_factors"]["regulation_coverage"]["rubric_level"])
    for cid, c in clients.items():
        docs = copy.deepcopy(manifest["clients"][cid]["documentary_factors"])
        if threat == "inflation" and (c["degraded"] or cid == target_low):
            for factor in FACTORS["D3"]+FACTORS["D5"]:
                docs[factor]["rubric_level"] = 5
                docs[factor]["evidence_ref"] = f"SIMULATED_FALSE_CLAIM:{cid}:{factor}"
                docs[factor]["descriptor"] = "Manipulated unverified claim, simulated verification failure"
                # Attacker has managed to mislead a reviewer. Verification is NOT a defense in this case.
        if threat == "stale" and (c["degraded"] or cid == target_low):
            docs["data_freshness"]["rubric_level"] = 1
            docs["data_freshness"]["evidence_ref"] = f"SIMULATED_STALE:{cid}"
        if threat == "inconsistent" and (c["degraded"] or cid == target_low):
            for factor in FACTORS["D3"]:
                docs[factor]["rubric_level"] = 5
                docs[factor]["evidence_ref"] = f"SIMULATED_HIGH_DOCUMENTATION:{cid}:{factor}"
        metrics = (adult_train_metrics(c["x_raw"],c["y_raw"],xreference) if dataset=="adult"
                   else fashion_train_metrics(c["x_raw"],c["y_raw"],xreference))
        metric_map[cid] = metrics
        decision = evaluate_client(docs, metrics, policy)
        decisions[cid] = decision
        records[cid] = docs
    return decisions, metric_map, records


class FederatedAdultPreprocessor:
    """Accepted-TRAIN-only statistics and accepted-TRAIN-only category vocabulary.

    Categories are hashed into a FIXED schema of 32 bins per field (configurable).
    This avoids a model-dimension mismatch between methods with different
    admitted cohorts while preventing any vocabulary fitted from TEST or
    rejected clients. Unknown TEST categories produce zeros, not fitted bins.
    Fixed hashing may produce collisions; these are documented and deterministic.

    Aggregate sufficient statistics simulate federated computation; they are
    NOT cryptographically secure aggregation or proof of practical privacy.
    """
    def __init__(self, cohort, hash_bins=32):
        self.numeric=list(ADULT_NUMERIC)
        self.categorical=[c for c in ADULT_COLUMNS if c not in self.numeric and c!="income"]
        self.hash_bins=hash_bins
        sums=np.zeros(len(self.numeric));sq=np.zeros(len(self.numeric));counts=np.zeros(len(self.numeric))
        self.vocabulary={c:set() for c in self.categorical}
        for cid in sorted(cohort):
            raw=cohort[cid]["x_raw"]
            z=self._numerical_with_schema_checks(raw)
            finite=np.isfinite(z)
            sums+=np.where(finite,z,0).sum(axis=0)
            sq+=np.where(finite,z*z,0).sum(axis=0)
            counts+=finite.sum(axis=0)
            for col in self.categorical:
                normalized=raw[col].fillna("MISSING").astype(str).str.strip().to_list()
                self.vocabulary[col].update(normalized)
        self.means=np.divide(sums,counts,out=np.zeros_like(sums),where=counts>0)
        variance=np.maximum(np.divide(sq,counts,out=np.zeros_like(sq),where=counts>0)-self.means**2,0)
        self.stds=np.sqrt(variance)
        self.stds[self.stds<1e-9]=1
        self.n_features=len(self.numeric)+len(self.categorical)*self.hash_bins

    def _numerical_with_schema_checks(self,x):
        """Schema checks are fixed BEFORE inspection of any split.

        Invalid numerical values become missing values for imputation; their
        original errors remain visible in the pre-admission D2 measurements.
        """
        z=x[self.numeric].apply(pd.to_numeric,errors="coerce").to_numpy(dtype=float)
        for j,name in enumerate(self.numeric):
            lo,hi=NUMERIC_RANGES[name]
            bad=(~np.isfinite(z[:,j])) | (z[:,j]<lo) | (z[:,j]>hi)
            z[bad,j]=np.nan
        return z

    def transform(self,x):
        z=self._numerical_with_schema_checks(x)
        z=np.where(np.isfinite(z),z,self.means)
        numeric=np.asarray((z-self.means)/self.stds,dtype=np.float32)
        categorical=np.zeros((len(x),len(self.categorical)*self.hash_bins),dtype=np.float32)
        for i,col in enumerate(self.categorical):
            vals=x[col].fillna("MISSING").astype(str).str.strip().to_numpy()
            permitted=self.vocabulary[col]
            for j,value in enumerate(vals):
                if value in permitted:
                    digest=hashlib.sha256((col+"::"+value).encode()).digest()
                    bin_id=int.from_bytes(digest[:8],"big") % self.hash_bins
                    categorical[j,i*self.hash_bins+bin_id]=1
        return np.hstack((numeric,categorical))


def prep_clients(dataset, clients, cohort_ids, reference_x, test_x, cfg=None):
    """Fit all model transforms only after cohort selection. TEST transform only."""
    out={}
    if dataset=="adult":
        subset={cid:clients[cid] for cid in cohort_ids}
        prep=FederatedAdultPreprocessor(subset,hash_bins=cfg.category_hash_bins if cfg else 32)
        xte=prep.transform(test_x)
        for cid in cohort_ids:
            out[cid]={"x":prep.transform(clients[cid]["x_raw"]),
                      "y":clients[cid]["y_raw"].astype(np.int64),"n":clients[cid]["n"]}
        # Cohort-level loss balancing computed only from participating TRAIN labels.
        train_y=np.concatenate([clients[cid]["y_raw"] for cid in cohort_ids])
        unique=np.unique(train_y)
        if not set([0,1]).issubset(set(unique.tolist())):
            raise ValueError("Selected cohort lacks both labels; training is infeasible")
        pos=int(np.sum(train_y==1));neg=int(np.sum(train_y==0))
        cap=cfg.class_weight_cap if cfg else 5.0
        pos_weight=min(cap,neg/max(pos,1)) if cfg is None or cfg.use_train_cohort_class_weight else 1.0
        for cid in out:out[cid]["positive_class_weight"]=float(pos_weight)
    else:
        def trans(x):
            v=np.asarray(x,dtype=np.float32)
            return np.clip(np.nan_to_num(v,nan=0.,posinf=255.,neginf=0.),0,255)[:,None,:,:]/255.
        xte=trans(test_x)
        for cid in cohort_ids:
            out[cid]={"x":trans(clients[cid]["x_raw"]),"y":clients[cid]["y_raw"].astype(np.int64),
                      "n":clients[cid]["n"],"positive_class_weight":1.0}
    return out,xte


class AdultModel(nn.Module):
    def __init__(self,d):
        super().__init__()
        self.m = nn.Sequential(nn.Linear(d,256),nn.ReLU(),nn.Linear(256,128),nn.ReLU(),
                               nn.Linear(128,64),nn.ReLU(),nn.Linear(64,32),nn.ReLU(),nn.Linear(32,1))
    def forward(self,x):return self.m(x).squeeze(-1)

class FashionModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.features=nn.Sequential(nn.Conv2d(1,32,3,padding=1),nn.ReLU(),nn.Conv2d(32,64,3,padding=1),
                                    nn.ReLU(),nn.MaxPool2d(2),nn.Conv2d(64,128,3,padding=1),nn.ReLU(),nn.MaxPool2d(2))
        self.classifier=nn.Sequential(nn.Flatten(),nn.Linear(128*7*7,256),nn.ReLU(),nn.Linear(256,10))
    def forward(self,x):return self.classifier(self.features(x))


def model_factory(dataset, n_features, device):
    model=AdultModel(n_features) if dataset=="adult" else FashionModel()
    return model.to(device)


def model_payload_bytes(model):
    return sum(t.numel()*t.element_size() for t in model.state_dict().values())


def split_fingerprint(xtr,ytr,xt,yt):
    """Bind cached results to the full, ordered TRAIN and held-out TEST data.

    TEST is hashed only for cache integrity; it is not used for D2, admission,
    loss weights, model selection, or fitting.
    """
    digest=hashlib.sha256()
    for tag,x,y in ((b'TRAIN',xtr,ytr),(b'TEST',xt,yt)):
        digest.update(tag)
        digest.update(np.asarray(pd.util.hash_pandas_object(x,index=False).to_numpy(),dtype=np.uint64).tobytes()
                      if isinstance(x,pd.DataFrame) else np.ascontiguousarray(x).tobytes())
        digest.update(np.ascontiguousarray(np.asarray(y,dtype=np.int64)).tobytes())
    return digest.hexdigest()


def init_model(dataset,n_features,seed,device):
    set_seed(seed)
    return model_factory(dataset,n_features,device)


def _criterion(pred, target, dataset, pos_weight=1.0):
    if dataset=="adult":return nn.functional.binary_cross_entropy_with_logits(
        pred,target.float(),pos_weight=torch.tensor(float(pos_weight),device=pred.device))
    return nn.functional.cross_entropy(pred,target.long())


def train_client(global_model, data, cfg, seed, r, cid, prox=False):
    """Exactly cfg.local_steps optimizer updates per selected client, replacement sampling."""
    local = copy.deepcopy(global_model)
    local.train()
    optim=torch.optim.Adam(local.parameters(),lr=cfg.lr)
    x,y = data["x"], data["y"]
    rng=np.random.default_rng(seed*1234567+r*971+int(cid[1:])*31)
    if not len(y):raise ValueError("Zero-size client")
    global_params=[t.detach().clone() for t in global_model.parameters()]
    for step in range(cfg.local_steps):
        ids=rng.integers(0,len(y),size=cfg.batch_size)
        xb=torch.as_tensor(x[ids],dtype=torch.float32,device=cfg.device)
        yb=torch.as_tensor(y[ids],dtype=torch.long,device=cfg.device)
        optim.zero_grad(set_to_none=True)
        logits=local(xb)
        loss=_criterion(logits,yb,cfg.dataset,data.get("positive_class_weight",1.0))
        if prox:
            loss += (cfg.fedprox_mu/2)*sum(((p-q)**2).sum() for p,q in zip(local.parameters(),global_params))
        loss.backward();optim.step()
    return {k:v.detach().cpu().clone() for k,v in local.state_dict().items()}


def agg_states(states, weights, rule, f=1):
    keys=tuple(states[0])
    for st in states:
        if tuple(st)!=keys:raise ValueError("Incompatible model states")
    if rule == "krum":
        n=len(states)
        if n < 2*f+3:raise ValueError(f"Krum with f={f} requires n >= {2*f+3} trained updates")
        flat=torch.stack([torch.cat([v.float().reshape(-1) for v in st.values()]) for st in states])
        dist=torch.cdist(flat,flat,p=2).square()
        # Exclude self; n-f-2 neighbors, smaller score wins.
        dist.fill_diagonal_(float("inf"))
        k=n-f-2
        scores=dist.topk(k,largest=False).values.sum(dim=1)
        winner=int(scores.argmin())
        return states[winner], [float(i==winner) for i in range(n)]
    if rule=="median":
        return {k:torch.stack([st[k].float() for st in states]).median(dim=0).values for k in keys}, [1/len(states)]*len(states)
    weights=np.asarray(weights,dtype=float)
    if not len(weights) or not np.isfinite(weights).all() or (weights<0).any() or weights.sum()<=0:
        raise ValueError("Invalid aggregation weights")
    weights=weights/weights.sum()
    new={key:sum(float(w)*st[key].float() for w,st in zip(weights,states)) for key in keys}
    return new, weights.tolist()


def local_loss(global_model, data, cfg, seed, r, cid):
    rng=np.random.default_rng(seed*211+r*101+int(cid[1:]))
    ids=rng.integers(0,len(data["y"]),size=min(cfg.batch_size,len(data["y"])))
    global_model.eval()
    with torch.no_grad():
        x=torch.as_tensor(data["x"][ids],dtype=torch.float32,device=cfg.device)
        y=torch.as_tensor(data["y"][ids],dtype=torch.long,device=cfg.device)
        return float(_criterion(global_model(x),y,cfg.dataset,data.get("positive_class_weight",1.0)).item())


def eval_model(model,x_test,y_test,cfg):
    model.eval()
    ps=[]
    with torch.no_grad():
        for start in range(0,len(y_test),512):
            v=torch.as_tensor(x_test[start:start+512],dtype=torch.float32,device=cfg.device)
            o=model(v)
            p=torch.sigmoid(o).cpu().numpy() if cfg.dataset=="adult" else torch.softmax(o,dim=1).cpu().numpy()
            ps.append(p)
    p=np.concatenate(ps,axis=0)
    yhat=(p>=.5).astype(int) if cfg.dataset=="adult" else p.argmax(axis=1)
    result={"accuracy":float(accuracy_score(y_test,yhat)),
            "macro_f1":float(f1_score(y_test,yhat,average="macro",zero_division=0)),
            "macro_recall":float(recall_score(y_test,yhat,average="macro",zero_division=0)),
            "balanced_accuracy":float(balanced_accuracy_score(y_test,yhat))}
    if cfg.dataset == "adult":
        tn,fp,fn,tp=confusion_matrix(y_test,yhat,labels=[0,1]).ravel()
        negative_rate=float(np.mean(np.asarray(y_test)==0))
        majority_class=int(negative_rate < 0.5)
        result.update({
            "majority_accuracy":max(negative_rate,1-negative_rate),
            "majority_class":majority_class,
            "positive_prevalence":float(np.mean(np.asarray(y_test)==1)),
            "predicted_positive_fraction":float(np.mean(yhat==1)),
            "positive_precision":float(precision_score(y_test,yhat,zero_division=0)),
            "positive_recall":float(recall_score(y_test,yhat,zero_division=0)),
            "tn":int(tn),"fp":int(fp),"fn":int(fn),"tp":int(tp),
            "pr_auc_average_precision":float(average_precision_score(y_test,p)),
            "is_majority_collapse":bool(np.all(yhat==majority_class))})
    try:
        result["auc"] = float(roc_auc_score(y_test,p)) if cfg.dataset=="adult" else float(roc_auc_score(y_test,p,average="macro",multi_class="ovr"))
    except ValueError:result["auc"] = float("nan")
    return result


def run_method(method, clients, decisions, cfg, xref, xtest, ytest, seed, policy=Policy(),
               shuffled_decisions=None, ablated_decisions=None, log_rounds=True):
    """Return final metrics & per-round ledger-like records for ONE method."""
    ids=sorted(clients)
    effective = ablated_decisions if method.startswith("NoDist_") else decisions
    if effective is None:raise ValueError("Missing ablation decisions")
    eligible=[i for i in ids if effective[i].accepted]
    if method=="RandomHPS":
        if shuffled_decisions is None:raise ValueError("Provide randomly permuted full-factor HPS decisions")
        eligible=[i for i in ids if shuffled_decisions[i].accepted]
    elif method=="D2_only":
        eligible=[i for i in ids if decisions[i].dimensions["D2"]>=policy.min_dimension
                  and all(decisions[i].factor_scores[f]>0 for f in FACTORS["D2"])]
    no_gate={"FedAvg_all","FedAvg_matched","FedProx","Median","Krum","PowerOfChoice"}
    allowed=ids if method in no_gate else eligible
    # The target is a TRAINING-SCHEDULE limit, never an HPS admission quota.
    # run_experiment adjusts cfg.per_round to the HPS-approved cohort count,
    # and matched baseline methods inherit the same effective round budget.
    target=(len(ids) if method=="FedAvg_all" else cfg.per_round)
    if len(allowed)<target:
        raise ValueError(f"{method}: eligible pool {len(allowed)} < matched round budget {target}; "
                         "no governance re-admission or artificial cohort expansion")
    # Participant-dependent imputer and scaler are fit once before rounds.
    prep, xte = prep_clients(cfg.dataset,clients,allowed,xref,xtest,cfg)
    n_features=next(iter(prep.values()))["x"].shape[1] if cfg.dataset=="adult" else None
    model=init_model(cfg.dataset,n_features,seed,cfg.device)
    upload=model_payload_bytes(model)
    history=[]; start=time.perf_counter()
    milestone=max(1,cfg.rounds//5)
    for r in range(1,cfg.rounds+1):
        rng=np.random.default_rng(seed*99991+r*1009)
        if method=="FedAvg_all":chosen=list(ids)
        elif method=="PowerOfChoice":
            d=min(len(ids),max(cfg.per_round,cfg.per_round*2))
            candidates=sorted(rng.choice(allowed,size=d,replace=False).tolist())
            # Probe candidate local TRAIN loss, do NOT train or upload unselected models.
            losses={c:local_loss(model,prep[c],cfg,seed,r,c) for c in candidates}
            chosen=sorted(candidates,key=lambda c:losses[c],reverse=True)[:cfg.per_round]
        else:
            chosen=sorted(rng.choice(allowed,size=target,replace=False).tolist())
        states=[train_client(model,prep[c],cfg,seed,r,c,prox=(method=="FedProx")) for c in chosen]
        if method in ("PWA", "NoDist_PWA") or (method=="WUPW" and r>cfg.warmup_rounds):
            weights=[prep[c]["n"] * max(0.,effective[c].hps) for c in chosen]
        elif method=="RandomHPS":
            weights=[prep[c]["n"] * max(0.,shuffled_decisions[c].hps) for c in chosen]
        else:
            weights=[prep[c]["n"] for c in chosen]
        rule="median" if method=="Median" else ("krum" if method=="Krum" else "average")
        new_state, actual_weights=agg_states(states,weights,rule,cfg.krum_f)
        model.load_state_dict(new_state)
        # Exactly target*local_steps actual optimizer updates; PoC additionally evaluates losses.
        audit_hps = shuffled_decisions if method == "RandomHPS" else effective
        weighted_hps=sum(actual_weights[j]*audit_hps[c].hps for j,c in enumerate(chosen))
        history.append({"round":r,"method":method,"selected_ids":chosen,
                        "weights":{c:actual_weights[j] for j,c in enumerate(chosen)},
                        "selected_clients":len(chosen),"optimizer_steps":len(chosen)*cfg.local_steps,
                        "model_upload_bytes":len(chosen)*upload,
                        "selection_probe_clients":len(candidates) if method=="PowerOfChoice" else 0,
                        "aggregation_weighted_hps":weighted_hps})
        if r == 1 or r == cfg.rounds or r % milestone == 0:
            print(f"      round {r:>3}/{cfg.rounds} | selected={len(chosen):>2} "
                  f"| cumulative steps={r*len(chosen)*cfg.local_steps:>5} "
                  f"| model upload payload={r*len(chosen)*upload/1e6:.2f} MB",flush=True)
    elapsed=time.perf_counter()-start
    result=eval_model(model,xte,ytest,cfg)
    result.update({"method":method,"seed":seed,"cohort_size":len(allowed),
                   "training_clients_per_round":target,"total_optimizer_steps":sum(h["optimizer_steps"] for h in history),
                   "upload_MB_payload":sum(h["model_upload_bytes"] for h in history)/1e6,
                   "estimated_energy_kwh":elapsed/3600*cfg.energy_power_kw_assumed,
                   "estimated_co2_kg":elapsed/3600*cfg.energy_power_kw_assumed*cfg.co2_kg_per_kwh_assumed,
                   "runtime_seconds":elapsed,
                   "train_only_positive_class_weight":prep[allowed[0]].get("positive_class_weight",1.0),
                   "mean_participant_hps":float(np.mean([(shuffled_decisions if method == "RandomHPS" else effective)[c].hps for c in allowed])),
                   "mean_aggregation_weighted_hps":float(np.mean([h["aggregation_weighted_hps"] for h in history]))})
    return result,history


def run_experiment(cfg, seed, loaded, outdir, evidence_seed=7702,
                   policy=Policy(), degraded=False, threat="none", methods=METHODS,
                   manifest_file=None):
    cfg.validate();policy.validate();set_seed(seed)
    outdir=Path(outdir);outdir.mkdir(parents=True,exist_ok=True)
    xtrain,ytrain,xref,xtest,ytest=split_global(cfg.dataset,loaded,cfg,seed)
    split_sha256 = split_fingerprint(xtrain,ytrain,xtest,ytest)
    ids=[f"C{i:03d}" for i in range(cfg.n_clients)]
    clients,corrupted,partition_diag=make_clients(cfg.dataset,xtrain,ytrain,ids,cfg,seed,degraded)
    manifest_path=Path(manifest_file or (outdir/f"manifest_{cfg.dataset}_K{cfg.n_clients}_E{evidence_seed}.json"))
    if manifest_path.exists():manifest=load_manifest(manifest_path)
    else:
        manifest=make_manifest(ids,evidence_seed)
        save_manifest(manifest,manifest_path)
    if set(manifest["clients"]) != set(ids):
        raise ValueError("Manifest IDs do not match; refuse to silently resample")
    decisions,metrics,records=assess_clients(cfg.dataset,clients,xref,manifest,policy,threat)
    ablated={cid:evaluate(decisions[cid].factor_scores,policy,ablate_distribution=True)
             for cid in ids}
    shuffled={}
    shuffler=np.random.default_rng(seed+837)
    order=list(shuffler.permutation(ids))
    for cid,source in zip(ids,order):
        shuffled[cid]=evaluate(decisions[source].factor_scores,policy)
    tag=f"{cfg.dataset}_seed{seed}_{'degraded' if degraded else 'clean'}_{threat}"
    decisions_path=outdir/f"decisions_{tag}.json"
    results_path=outdir/f"results_{tag}.csv"
    rounds_path=outdir/f"rounds_{tag}.json"
    failure_path=outdir/f"failed_{tag}.json"
    hps_admitted_ids=[cid for cid in ids if decisions[cid].accepted]
    n_hps_admitted=len(hps_admitted_ids)
    # Governance always admits precisely the clients who passed. There is NO
    # minimum-10 admission requirement. Adapt the matched optimization budget
    # only AFTER admission, never the policy or its thresholds.
    matched_training_possible=n_hps_admitted>0
    effective_m=min(cfg.per_round,n_hps_admitted) if matched_training_possible else cfg.per_round
    execution_cfg=replace(cfg,per_round=effective_m)
    print(f"    GOVERNANCE: ACCEPT={n_hps_admitted}/{len(ids)}  REJECT={len(ids)-n_hps_admitted}  "
          f"SCHEDULE: target={cfg.per_round}, effective={effective_m if matched_training_possible else 0} "
          f"{'(matched FL possible)' if matched_training_possible else '(ABSTAIN: no approved clients)'}",flush=True)
    print_budget_explanation(cfg,n_hps_admitted)
    decision_payload={"dataset":cfg.dataset,"seed":seed,"code_version":RUN_CODE_VERSION,
        "global_train_test_sha256":split_sha256,
        "degraded_ids":corrupted,"synthetic_documentary_evidence":True,
        "manifest_sha256":manifest["sha256"],"policy":asdict(policy),
        "training_config":asdict(cfg),"partition_diagnostics":partition_diag,
        "hps_accepted_clients":n_hps_admitted,
        "hps_training_possible":matched_training_possible,
        "effective_matched_trainers_per_round":effective_m if matched_training_possible else 0,
        "budget_matching_rule":"min(nominal_per_round, HPS_admitted_count); no forced admission",
        "decisions":{c:{**decisions[c].record(),"train_d2_raw_metrics":metrics[c],
                         "degraded":clients[c]["degraded"]} for c in ids}}
    # Condition-level checkpoints: only reload partial results if exact run
    # configuration, policy, seed, evidence manifest and decisions match.
    results=[];history=[];failed={}
    resume_ok=False
    if decisions_path.exists() and failure_path.exists() and rounds_path.exists() and results_path.exists():
        try:
            prior=json.loads(decisions_path.read_text())
            prior_results=pd.read_csv(results_path) if results_path.stat().st_size else pd.DataFrame()
            prior_failures=json.loads(failure_path.read_text())
            prior_history=json.loads(rounds_path.read_text())
            resume_ok=(_canonical(prior)==_canonical(decision_payload) and isinstance(prior_failures,dict)
                       and isinstance(prior_history,list) and
                       set(prior_results.get('method',pd.Series(dtype=str)))
                       .isdisjoint(set(prior_failures)))
            if resume_ok:
                results=prior_results.to_dict('records')
                failed=prior_failures
                history=prior_history
                print(f"METHOD-CHECKPOINT RESUME {tag}: {len(results)} successful, "
                      f"{len(failed)} recorded failures",flush=True)
        except Exception:
            resume_ok=False
    if not resume_ok:
        # Atomic replace for metadata; if the code version/config changes,
        # discard old checkpoints rather than risking cross-run contamination.
        tmp=decisions_path.with_suffix('.tmp')
        tmp.write_text(json.dumps(decision_payload,indent=2,default=str),encoding='utf-8')
        tmp.replace(decisions_path)

    def checkpoint():
        # Atomic file replacement avoids partial writes on Colab interruption.
        csv_tmp=results_path.with_suffix('.csv.tmp')
        pd.DataFrame(results,columns=['method'] if not results else None).to_csv(csv_tmp,index=False)
        csv_tmp.replace(results_path)
        for target,obj in ((rounds_path,history),(failure_path,failed)):
            temp=target.with_suffix('.json.tmp')
            temp.write_text(json.dumps(obj,indent=2,default=str),encoding='utf-8')
            temp.replace(target)
    if not resume_ok:checkpoint()
    completed={r['method'] for r in results}|set(failed)
    for index,method in enumerate(methods, start=1):
        if method in completed:
            print(f"    [{index:02d}/{len(methods):02d}] {method:<18} REUSED checked checkpoint",flush=True)
            continue
        print(f"    [{index:02d}/{len(methods):02d}] {method:<18} START",flush=True)
        try:
            result,log=run_method(method,clients,decisions,execution_cfg,xref,xtest,ytest,seed,policy,shuffled,ablated)
            result.update({"dataset":cfg.dataset,"degraded":degraded,"threat":threat,
                           "manifest_sha256":manifest["sha256"],
                           "nominal_trainers_per_round":cfg.per_round,
                           "hps_approved_clients":n_hps_admitted,
                           "hps_matched_budget_available":matched_training_possible,
                           "budget_matched_to_hps":bool(matched_training_possible and method != 'FedAvg_all'),
                           "effective_reference_trainers_per_round":effective_m if matched_training_possible else 0})
            if not matched_training_possible and method not in {'FedAvg_all','FedAvg_matched','FedProx','Median','Krum','PowerOfChoice'}:
                raise AssertionError('Governance-approved FL cannot proceed with zero accepted clients')
            expected=execution_cfg.rounds * result['training_clients_per_round'] * execution_cfg.local_steps
            if int(result['total_optimizer_steps'])!=int(expected):
                raise AssertionError(f"Training budget mismatch in {method}: expected {expected}, got {result['total_optimizer_steps']}")
            results.append(result);history.extend(log)
            print(f"    [{index:02d}/{len(methods):02d}] {method:<18} DONE  "
                  f"Acc={result['accuracy']:.4f}  MacroF1={result['macro_f1']:.4f}  "
                  f"steps={result['total_optimizer_steps']}  "
                  f"runtime={result['runtime_seconds']:.1f}s",flush=True)
        except (ValueError,RuntimeError) as exc:
            failed[method]=str(exc)
            print(f"    [{index:02d}/{len(methods):02d}] {method:<18} INFEASIBLE/FAILED: {exc}",flush=True)
        checkpoint()
    return results,failed


from scipy.stats import t as tdist, ttest_rel

def summarize(df, outdir):
    if df.empty:return
    keys=["dataset","alpha","degraded","threat","method","manifest_sha256"]
    numeric=[c for c in ["accuracy","macro_f1","macro_recall","auc","balanced_accuracy",
             "pr_auc_average_precision", "positive_precision", "positive_recall",
             "predicted_positive_fraction", "majority_accuracy", "runtime_seconds",
             "estimated_energy_kwh", "estimated_co2_kg",
             "upload_MB_payload", "total_optimizer_steps", "cohort_size",
             "training_clients_per_round", "mean_participant_hps",
             "mean_aggregation_weighted_hps"] if c in df.columns]
    rows=[]
    for group,part in df.groupby(keys,dropna=False):
        row=dict(zip(keys,group));row["n_seeds"]=len(part)
        for col in numeric:
            row[col+"_mean"]=float(part[col].mean())
            row[col+"_sd"]=float(part[col].std(ddof=1)) if len(part)>1 else float("nan")
        rows.append(row)
    summary_df=pd.DataFrame(rows)
    summary_df.to_csv(outdir/"summary.csv",index=False)
    def fmt(row,metric):
        if metric+"_mean" not in row or pd.isna(row[metric+"_mean"]): return "N/A"
        mu=row[metric+"_mean"]; sd=row[metric+"_sd"]
        digits=7 if metric in ("estimated_energy_kwh", "estimated_co2_kg") else 4
        return f"{mu:.{digits}f} ± {sd:.{digits}f}" if np.isfinite(sd) else f"{mu:.{digits}f} [single run]"
    table=[]
    for _, row in summary_df.iterrows():
        table.append({"dataset":row["dataset"],"alpha":row["alpha"],"degraded":row["degraded"],
                      "threat":row["threat"],"method":row["method"],
                      "seeds":int(row["n_seeds"]),
                      "Accuracy":fmt(row,"accuracy"),"Macro F1":fmt(row,"macro_f1"),
                      "Balanced Accuracy":fmt(row,"balanced_accuracy"),
                      "ROC-AUC":fmt(row,"auc"),"PR-AUC (AP)":fmt(row,"pr_auc_average_precision"),
                      "Positive Recall":fmt(row,"positive_recall"),
                      "Client cohort":fmt(row,"cohort_size"),
                      "Trained clients / round":fmt(row,"training_clients_per_round"),
                      "Total optimizer steps":fmt(row,"total_optimizer_steps"),
                      "Time (s)":fmt(row,"runtime_seconds"),
                      "Estimated energy (kWh)":fmt(row,"estimated_energy_kwh"),
                      "Estimated CO2 (kg)":fmt(row,"estimated_co2_kg"),
                      "Average admitted HPS":fmt(row,"mean_participant_hps"),
                      "Aggregation weighted HPS":fmt(row,"mean_aggregation_weighted_hps"),
                      "Upload MB (payload)":fmt(row,"upload_MB_payload")})
    pd.DataFrame(table).to_csv(outdir/"journal_descriptive_table.csv",index=False)
    # Paired differences: compare equal-budget methods to FedAvg_matched.
    paired=[]
    for (ds,alpha,degr,th,manifest),part in df.groupby(["dataset","alpha","degraded","threat","manifest_sha256"],dropna=False):
        baselines=part[part.method=="FedAvg_matched"].set_index("seed")
        for method,grp in part.groupby("method"):
            if method in ("FedAvg_matched","FedAvg_all"):continue
            common=sorted(set(baselines.index)&set(grp.seed))
            if len(common)<2:continue
            matched=grp.set_index("seed")
            delta=(matched.loc[common,"accuracy"]-baselines.loc[common,"accuracy"]).to_numpy()
            mean=float(delta.mean());sem=float(delta.std(ddof=1)/np.sqrt(len(delta)))
            ci=float(tdist.ppf(.975,len(delta)-1)*sem)
            f1delta=(matched.loc[common,"macro_f1"]-baselines.loc[common,"macro_f1"]).to_numpy()
            f1mean=float(f1delta.mean());f1ci=float(tdist.ppf(.975,len(common)-1)*f1delta.std(ddof=1)/np.sqrt(len(common)))
            def paired_p(delta):
                if not np.isfinite(delta).all() or len(delta)<3 or np.isclose(delta.std(ddof=1),0):
                    return float('nan')  # t-test undefined when differences have zero variance
                return float(ttest_rel(delta, np.zeros_like(delta)).pvalue)
            paired.append({"dataset":ds,"alpha":alpha,"degraded":degr,"threat":th,"manifest_sha256":manifest,
                           "method":method,"n_pairs":len(common),"accuracy_delta":mean,
                           "ci95_lower":mean-ci,"ci95_upper":mean+ci,
                           "accuracy_paired_t_p_unadjusted":paired_p(delta),
                           "macro_f1_delta":f1mean,"macro_f1_ci95_lower":f1mean-f1ci,
                           "macro_f1_ci95_upper":f1mean+f1ci,
                           "macro_f1_paired_t_p_unadjusted":paired_p(f1delta)})
    pairs_df=pd.DataFrame(paired)
    if not pairs_df.empty:
        # Holm multiplicity adjustment within each dataset/alpha/degradation/
        # threat/manifest combination, separately for each endpoint.
        family_cols=['dataset','alpha','degraded','threat','manifest_sha256']
        for raw in ('accuracy_paired_t_p_unadjusted','macro_f1_paired_t_p_unadjusted'):
            adj=raw.replace('_unadjusted','_holm')
            pairs_df[adj]=float('nan')
            for _,part in pairs_df.groupby(family_cols,dropna=False):
                valid=part[raw].dropna().sort_values()
                if valid.empty: continue
                m=len(valid)
                unbounded=np.array([val*(m-j) for j,val in enumerate(valid.to_numpy())])
                corrected=np.minimum(1,np.maximum.accumulate(unbounded))
                pairs_df.loc[valid.index,adj]=corrected
    pairs_df.to_csv(outdir/"paired_vs_FedAvg_matched.csv",index=False)
    robustness=[]
    for (ds,alpha,th,method,manifest),part in df.groupby(
            ["dataset","alpha","threat","method","manifest_sha256"],dropna=False):
        clean=part[part.degraded==False].set_index("seed")
        corrupt=part[part.degraded==True].set_index("seed")
        shared=sorted(set(clean.index)&set(corrupt.index))
        if len(shared)<2: continue
        row={"dataset":ds,"alpha":alpha,"threat":th,"method":method,
             "manifest_sha256":manifest,"n_paired_seeds":len(shared)}
        for metric in ("accuracy","macro_f1","balanced_accuracy","positive_recall","pr_auc_average_precision"):
            if metric not in part.columns:continue
            vals=(corrupt.loc[shared,metric]-clean.loc[shared,metric]).to_numpy(dtype=float)
            vals=vals[np.isfinite(vals)]
            row[f"degraded_minus_clean_{metric}"]=float(vals.mean()) if len(vals) else float('nan')
            row[f"paired_n_{metric}"]=len(vals)
            row[f"paired_sd_{metric}"]=float(vals.std(ddof=1)) if len(vals)>1 else float('nan')
        robustness.append(row)
    pd.DataFrame(robustness).to_csv(outdir/"paired_clean_vs_degraded.csv",index=False)
    # An incomplete five-seed group is descriptive only; NEVER label it an
    # adequately replicated journal result, even if one seed looks favorable.
    summary_df["complete_five_seeds"] = summary_df["n_seeds"].eq(5)
    summary_df.to_csv(outdir/"summary.csv",index=False)
    (outdir/"INCOMPLETE_GROUPS.csv").write_text(
        summary_df.loc[~summary_df.complete_five_seeds].to_csv(index=False),encoding='utf-8')



# ============================================================================
# PRESENTATION ONLY — PHASE AND RUN REPORTING
# ============================================================================
def phase(title, detail="", step=None, total=None):
    prefix=f"PHASE {step}/{total}" if step is not None else "PHASE"
    bar="="*92
    print(f"\n{bar}\n{prefix}  |  {title}\n{bar}",flush=True)
    if detail:print(detail,flush=True)


def condition_banner(index,total,alpha,seed,degraded,threat="none",schedule=""):
    print("\n"+"-"*92,flush=True)
    print(f"RUN {index:02d}/{total:02d}  |  alpha={alpha}  seed={seed}  "
          f"condition={'DEGRADED' if degraded else 'CLEAN'}  threat={threat}",flush=True)
    if schedule:print("TRAINING SCHEDULE | "+schedule,flush=True)
    print("-"*92,flush=True)


def condition_table(rows, failure_map, alpha=None,seed=None,degraded=None, threat="none"):
    """Display only one complete condition, never silently pool seeds."""
    if rows:
        df=pd.DataFrame(rows)
        cols=["method","accuracy","macro_f1","balanced_accuracy","auc",
              "pr_auc_average_precision","positive_recall","runtime_seconds",
              "estimated_energy_kwh","estimated_co2_kg","upload_MB_payload",
              "cohort_size","training_clients_per_round","total_optimizer_steps",
              "budget_matched_to_hps"]
        print("\n  PERFORMANCE & UTILITY  |  individual run (NOT mean ± SD)",flush=True)
        display(df[[c for c in cols if c in df.columns]].sort_values('method').round(5).reset_index(drop=True))
    if failure_map:
        print("  EXPLICITLY INFEASIBLE / FAILED METHODS",flush=True)
        display(pd.DataFrame([{'method':k,'reason':v} for k,v in failure_map.items()]))
    if not rows and not failure_map:print("No results recorded for this condition.",flush=True)


def print_budget_explanation(cfg,actual_hps_accepted=None):
    accepted=actual_hps_accepted
    eff=(min(cfg.per_round,accepted) if accepted is not None and accepted>0 else (cfg.per_round if accepted is None else 0))
    print(f"  Governance approved : {accepted if accepted is not None else 'assess first'} (0..{cfg.n_clients}; NO admission quota)",flush=True)
    print(f"  Nominal per round    : {cfg.per_round}",flush=True)
    print(f"  Effective per round  : {eff} for matched methods",flush=True)
    print(f"  Planned matched steps: {cfg.rounds} rounds x {eff} clients x {cfg.local_steps} steps = {cfg.rounds*eff*cfg.local_steps}",flush=True)
    print(f"  Batch size           : {cfg.batch_size}; mini-batch sampling WITH replacement",flush=True)
    print(f"  FedAvg_all           : {cfg.rounds} x {cfg.n_clients} x {cfg.local_steps} = {cfg.rounds*cfg.n_clients*cfg.local_steps} (UNMATCHED)",flush=True)
    print("  PoC selection probes are additional forward passes, not optimizer steps.",flush=True)

# ==========================================================================
# EXPERIMENT CONFIGURATION — FASHION-MNIST
# ==========================================================================
RUN_CODE_VERSION = "TADP-HPS28/FASHION-TARGETED-VALIDATION-V5/2026-10-17"
# TARGETED secondary cross-modal validation, NOT the full 13-method, 25-round benchmark.
# Preserves: 50 clients; 5 paired seeds; alpha 0.5/0.1; CLEAN/DEGRADED;
# matched participation 10/round; 6 methods.
# Adjusted: 15 rounds, 10 optimizer steps/client, 3 warm-up rounds.
# Distinct OUTPUT and RUN_CODE_VERSION prevent mixing with the full benchmark.
PILOT = False
ROOT = Path('/content') if Path('/content').exists() else Path.cwd()
# Outputs/checkpoints are stored in mounted Google Drive; raw dataset files stay local.
if Path('/content').exists():
    DRIVE_ROOT = Path('/content/drive/MyDrive')
    if not DRIVE_ROOT.is_dir():
        raise RuntimeError('Google Drive is not mounted. In a Colab cell first run: from google.colab import drive; drive.mount("/content/drive")')
    OUTPUT_ROOT = DRIVE_ROOT / 'TADP_HPS28_V5'
else:
    OUTPUT_ROOT = ROOT
OUTPUT = OUTPUT_ROOT/'TADP_HPS28_Fashion_Targeted_Validation_v5'
OUTPUT.mkdir(parents=True,exist_ok=True)

def run_study():
    phase('EXPERIMENT OVERVIEW',
          'Fashion-MNIST TARGETED VALIDATION | 5 seeds | fixed official TRAIN/TEST | simulated evidence',1,5)
    seeds=[42] if PILOT else [42,43,44,45,46]
    alphas=[0.5,0.1]
    count=6 if PILOT else 50
    rounds=5 if PILOT else 15
    m=3 if PILOT else 10
    steps=2 if PILOT else 10
    cfg0=Config(dataset='fashion',n_clients=count,rounds=rounds,per_round=m,
                local_steps=steps,alpha=0.5,batch_size=32 if PILOT else 64,
                warmup_rounds=2 if PILOT else 3,
                device='cpu' if PILOT else ('cuda' if torch.cuda.is_available() else 'cpu'))
    loaded=load_dataset('fashion',fashion_root=str(ROOT/'data'),download=True)
    tr,ytr,_,te,yte=split_global('fashion',loaded,cfg0,42)
    phase('DATA ISOLATION',
          f'Official train set {len(ytr):,}; held-out test set {len(yte):,}; no test-derived HPS or preprocessing.',2,5)
    phase('GOVERNANCE ADMISSION',
          'No quota: HPS may approve 0..50 clients; training budget adapts only after approval.',3,5)
    phase('PAIRED FL RUNS / LIVE UTILITY TABLES',
          f'5 seeds x 2 alpha x clean/degraded; CPU/GPU={cfg0.device}',4,5)
    all_rows=[];failures=[];run_index=0
    total=len(alphas)*len(seeds)*2
    methods=('FedAvg_matched','HPG','PWA','WUPW','D2_only','NoDist_HPG')
    for alpha in alphas:
        cfg=replace(cfg0,alpha=alpha)
        for seed in seeds:
            for degraded in (False,True):
                run_index+=1
                condition_banner(run_index,total,alpha,seed,degraded)
                destination=OUTPUT/f'fashion_alpha{alpha}'
                rows,bad=run_experiment(cfg,seed,loaded,destination,
                                        evidence_seed=7755,degraded=degraded,methods=methods)
                for row in rows:row['alpha']=alpha
                all_rows.extend(rows)
                failures.extend([{'alpha':alpha,'seed':seed,'degraded':degraded,'method':meth,'reason':why}
                                 for meth,why in bad.items()])
                condition_table(rows,bad,alpha,seed,degraded)
                pd.DataFrame(all_rows).to_csv(OUTPUT/'FASHION_PARTIAL_RESULTS.csv',index=False)
                pd.DataFrame(failures).to_csv(OUTPUT/'FASHION_PARTIAL_FAILURES.csv',index=False)
                print('  CHECKPOINT:',OUTPUT/'FASHION_PARTIAL_RESULTS.csv',flush=True)
    phase('FINAL SUMMARY',
          'Five-seed mean ± SD and paired comparisons; report infeasible methods explicitly.',5,5)
    if all_rows:
        df=pd.DataFrame(all_rows)
        df.to_csv(OUTPUT/'fashion_HPS28_all.csv',index=False)
        summarize(df,OUTPUT)
        print('  FINAL JOURNAL TABLE:',OUTPUT/'journal_descriptive_table.csv',flush=True)
        display(pd.read_csv(OUTPUT/'journal_descriptive_table.csv'))
    print('Study output directory:',OUTPUT,flush=True)

if __name__ == '__main__':
    run_study()



PHASE 1/5  |  EXPERIMENT OVERVIEW
Fashion-MNIST TARGETED VALIDATION | 5 seeds | fixed official TRAIN/TEST | simulated evidence


100%|██████████| 26.4M/26.4M [00:02<00:00, 11.3MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 174kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.18MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 13.0MB/s]


PHASE 2/5  |  DATA ISOLATION
Official train set 60,000; held-out test set 10,000; no test-derived HPS or preprocessing.

PHASE 3/5  |  GOVERNANCE ADMISSION
No quota: HPS may approve 0..50 clients; training budget adapts only after approval.

PHASE 4/5  |  PAIRED FL RUNS / LIVE UTILITY TABLES
5 seeds x 2 alpha x clean/degraded; CPU/GPU=cuda

--------------------------------------------------------------------------------------------
RUN 01/20  |  alpha=0.5  seed=42  condition=CLEAN  threat=none
--------------------------------------------------------------------------------------------


    GOVERNANCE: ACCEPT=42/50  REJECT=8  SCHEDULE: target=10, effective=10 (matched FL possible)
  Governance approved : 42 (0..50; NO admission quota)
  Nominal per round    : 10
  Effective per round  : 10 for matched methods
  Planned matched steps: 15 rounds x 10 clients x 10 steps = 1500
  Batch size           : 64; mini-batch sampling WITH replacement
  FedAvg_all           : 15 x 50 x 10 = 7500 (UNMATCHED)
  PoC selection probes are additional forward passes, not optimizer steps.
    [01/06] FedAvg_matched     START
      round   1/15 | selected=10 | cumulative steps=  100 | model upload payload=68.05 MB
      round   3/15 | selected=10 | cumulative steps=  300 | model upload payload=204.14 MB
      round   6/15 | selected=10 | cumulative steps=  600 | model upload payload=408.27 MB
      round   9/15 | selected=10 | cumulative steps=  900 | model upload payload=612.41 MB
      round  12/15 | selected=10 | cumulative steps= 1200 | model upload payload=816.54 MB
      round  15/15

,method,accuracy,macro_f1,balanced_accuracy,auc,runtime_seconds,estimated_energy_kwh,estimated_co2_kg,upload_MB_payload,cohort_size,training_clients_per_round,total_optimizer_steps,budget_matched_to_hps
0,D2_only,0.7891,0.76645,0.7891,0.97583,10.41358,0.00029,0.00012,1020.678,50,10,1500,True
1,FedAvg_matched,0.7863,0.76240,0.7863,0.97572,12.71339,0.00035,0.00014,1020.678,50,10,1500,True
2,HPG,0.7548,0.73039,0.7548,0.97370,9.97136,0.00028,0.00011,1020.678,42,10,1500,True
3,NoDist_HPG,0.7554,0.73067,0.7554,0.97395,9.98537,0.00028,0.00011,1020.678,42,10,1500,True
4,PWA,0.7442,0.71429,0.7442,0.97351,10.02876,0.00028,0.00011,1020.678,42,10,1500,True
5,WUPW,0.7415,0.71046,0.7415,0.97305,9.82596,0.00027,0.00011,1020.678,42,10,1500,True


...

  CHECKPOINT: /content/drive/MyDrive/TADP_HPS28_V5/TADP_HPS28_Fashion_Targeted_Validation_v5/FASHION_PARTIAL_RESULTS.csv

--------------------------------------------------------------------------------------------
RUN 20/20  |  alpha=0.1  seed=46  condition=DEGRADED  threat=none
--------------------------------------------------------------------------------------------
    GOVERNANCE: ACCEPT=28/50  REJECT=22  SCHEDULE: target=10, effective=10 (matched FL possible)
  Governance approved : 28 (0..50; NO admission quota)
  Nominal per round    : 10
  Effective per round  : 10 for matched methods
  Planned matched steps: 15 rounds x 10 clients x 10 steps = 1500
  Batch size           : 64; mini-batch sampling WITH replacement
  FedAvg_all           : 15 x 50 x 10 = 7500 (UNMATCHED)
  PoC selection probes are additional forward passes, not optimizer steps.
    [01/06] FedAvg_matched     START
      round   1/15 | selected=10 | cumulative steps=  100 | model upload payload=68.05 MB
      r

,method,accuracy,macro_f1,balanced_accuracy,auc,runtime_seconds,estimated_energy_kwh,estimated_co2_kg,upload_MB_payload,cohort_size,training_clients_per_round,total_optimizer_steps,budget_matched_to_hps
0,D2_only,0.5615,0.49574,0.5615,0.93499,10.35301,0.00029,0.00012,1020.678,35,10,1500,True
1,FedAvg_matched,0.5131,0.41228,0.5131,0.96087,10.39544,0.00029,0.00012,1020.678,50,10,1500,True
2,HPG,0.6014,0.51902,0.6014,0.94228,10.47660,0.00029,0.00012,1020.678,28,10,1500,True
3,NoDist_HPG,0.5994,0.51729,0.5994,0.94307,10.27045,0.00029,0.00011,1020.678,28,10,1500,True
4,PWA,0.5858,0.50326,0.5858,0.94219,10.56818,0.00029,0.00012,1020.678,28,10,1500,True
5,WUPW,0.5928,0.51536,0.5928,0.93826,10.44776,0.00029,0.00012,1020.678,28,10,1500,True


  CHECKPOINT: /content/drive/MyDrive/TADP_HPS28_V5/TADP_HPS28_Fashion_Targeted_Validation_v5/FASHION_PARTIAL_RESULTS.csv

PHASE 5/5  |  FINAL SUMMARY
Five-seed mean ± SD and paired comparisons; report infeasible methods explicitly.
  FINAL JOURNAL TABLE: /content/drive/MyDrive/TADP_HPS28_V5/TADP_HPS28_Fashion_Targeted_Validation_v5/journal_descriptive_table.csv


,dataset,alpha,degraded,threat,method,seeds,Accuracy,Macro F1,Balanced Accuracy,ROC-AUC,...,Positive Recall,Client cohort,Trained clients / round,Total optimizer steps,Time (s),Estimated energy (kWh),Estimated CO2 (kg),Average admitted HPS,Aggregation weighted HPS,Upload MB (payload)
0,fashion,0.1,False,none,D2_only,5,0.5763 ± 0.0708,0.5123 ± 0.0912,0.5763 ± 0.0708,0.9550 ± 0.0137,...,NaN,50.0000 ± 0.0000,10.0000 ± 0.0000,1500.0000 ± 0.0000,10.3915 ± 0.0416,0.0002887 ± 0.0000012,0.0001155 ± 0.0000005,3.8429 ± 0.0017,3.7801 ± 0.1623,1020.6780 ± 0.0000
1,fashion,0.1,False,none,FedAvg_matched,5,0.5776 ± 0.0705,0.5138 ± 0.0890,0.5776 ± 0.0705,0.9545 ± 0.0148,...,NaN,50.0000 ± 0.0000,10.0000 ± 0.0000,1500.0000 ± 0.0000,10.3777 ± 0.0218,0.0002883 ± 0.0000006,0.0001153 ± 0.0000002,3.8429 ± 0.0017,3.7801 ± 0.1623,1020.6780 ± 0.0000
2,fashion,0.1,False,none,HPG,5,0.5013 ± 0.1129,0.4222 ± 0.1457,0.5013 ± 0.1129,0.9425 ± 0.0195,...,NaN,42.0000 ± 0.0000,10.0000 ± 0.0000,1500.0000 ± 0.0000,10.4584 ± 0.0306,0.0002905 ± 0.0000008,0.0001162 ± 0.0000003,4.2503 ± 0.0013,4.2449 ± 0.0776,1020.6780 ± 0.0000
3,fashion,0.1,False,none,NoDist_HPG,5,0.5071 ± 0.1012,0.4329 ± 0.1273,0.5071 ± 0.1012,0.9434 ± 0.0185,...,NaN,42.0000 ± 0.0000,10.0000 ± 0.0000,1500.0000 ± 0.0000,10.3086 ± 0.0461,0.0002864 ± 0.0000013,0.0001145 ± 0.0000005,4.3059 ± 0.0000,4.3016 ± 0.0771,1020.6780 ± 0.0000
4,fashion,0.1,False,none,PWA,5,0.5112 ± 0.0998,0.4376 ± 0.1194,0.5112 ± 0.0998,0.9444 ± 0.0180,...,NaN,42.0000 ± 0.0000,10.0000 ± 0.0000,1500.0000 ± 0.0000,10.5741 ± 0.0138,0.0002937 ± 0.0000004,0.0001175 ± 0.0000002,4.2503 ± 0.0013,4.2747 ± 0.0735,1020.6780 ± 0.0000
5,fashion,0.1,False,none,WUPW,5,0.5133 ± 0.1036,0.4405 ± 0.1321,0.5133 ± 0.1036,0.9427 ± 0.0198,...,NaN,42.0000 ± 0.0000,10.0000 ± 0.0000,1500.0000 ± 0.0000,10.4725 ± 0.0497,0.0002909 ± 0.0000014,0.0001164 ± 0.0000006,4.2503 ± 0.0013,4.2704 ± 0.0754,1020.6780 ± 0.0000
6,fashion,0.1,True,none,D2_only,5,0.5694 ± 0.1008,0.5202 ± 0.1157,0.5694 ± 0.1008,0.9498 ± 0.0194,...,NaN,35.0000 ± 0.0000,10.0000 ± 0.0000,1500.0000 ± 0.0000,10.3621 ± 0.0224,0.0002878 ± 0.0000006,0.0001151 ± 0.0000002,3.8158 ± 0.1077,3.7968 ± 0.2713,1020.6780 ± 0.0000
7,fashion,0.1,True,none,FedAvg_matched,5,0.5576 ± 0.0698,0.4932 ± 0.0856,0.5576 ± 0.0698,0.9513 ± 0.0186,...,NaN,50.0000 ± 0.0000,10.0000 ± 0.0000,1500.0000 ± 0.0000,10.4133 ± 0.0376,0.0002893 ± 0.0000010,0.0001157 ± 0.0000004,3.7778 ± 0.0012,3.7190 ± 0.1602,1020.6780 ± 0.0000
8,fashion,0.1,True,none,HPG,5,0.5649 ± 0.1412,0.4994 ± 0.1908,0.5649 ± 0.1412,0.9401 ± 0.0363,...,NaN,29.2000 ± 1.3038,10.0000 ± 0.0000,1500.0000 ± 0.0000,10.4603 ± 0.0181,0.0002906 ± 0.0000005,0.0001162 ± 0.0000002,4.2485 ± 0.0322,4.2709 ± 0.0826,1020.6780 ± 0.0000
9,fashion,0.1,True,none,NoDist_HPG,5,0.5632 ± 0.1362,0.4991 ± 0.1861,0.5632 ± 0.1362,0.9420 ± 0.0329,...,NaN,29.2000 ± 1.3038,10.0000 ± 0.0000,1500.0000 ± 0.0000,10.2847 ± 0.0152,0.0002857 ± 0.0000004,0.0001143 ± 0.0000002,4.3048 ± 0.0325,4.3263 ± 0.0819,1020.6780 ± 0.0000


Study output directory: /content/drive/MyDrive/TADP_HPS28_V5/TADP_HPS28_Fashion_Targeted_Validation_v5


In [7]:


from google.colab import files
from pathlib import Path
import shutil

# Results stored in Google Drive
results_dir = Path(
    "/content/drive/MyDrive/TADP_HPS28_V5/"
    "TADP_HPS28_Fashion_Targeted_Validation_v5"
)

# Verify results exist
if not results_dir.is_dir():
    raise FileNotFoundError(
        f"Results folder not found: {results_dir}"
    )

# Create ZIP containing all experiment results
zip_path = shutil.make_archive(
    "/content/Fashion_MNIST_HPS28_V5_Results",
    "zip",
    root_dir=results_dir.parent,
    base_dir=results_dir.name
)

print("Results ZIP created:", zip_path)

# Automatically trigger download to your computer
files.download(zip_path)



Results ZIP created: /content/Fashion_MNIST_HPS28_V5_Results.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>